# Autism Screening (UCI AQ-10 Adult)

The UCI Autism Screening Adult dataset has 704 responses to the AQ-10 questionnaire plus age, gender, ethnicity, and a few other demographics. The label is the screening *outcome*, not a clinical diagnosis — important distinction. So really this is "can we predict the screening tool's own output from its inputs," which mostly tests whether ML can rederive a scoring rule.

I'm doing this on purpose as a sanity-check exercise. If my model can't recover an explicit scoring rule, something is broken in my pipeline. I expect very high accuracy and that's a feature of the dataset, not a model achievement.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score

RNG = np.random.default_rng(2)
DATA_DIR = Path("../../data/autism")

In [2]:
# I'll try to load the UCI ARFF/CSV; if not present, synthesize a small dataset that
# mimics the scoring rule (sum of 10 binary items, >=6 -> positive).
csv_path = DATA_DIR / "autism_screening.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    source = "UCI AQ-10"
else:
    n = 800
    items = RNG.integers(0, 2, size=(n, 10))
    age = RNG.integers(17, 65, size=n)
    gender = RNG.choice(["m", "f"], size=n)
    jaundice = RNG.choice(["yes", "no"], size=n, p=[0.15, 0.85])
    cols = [f"A{i+1}_Score" for i in range(10)]
    df = pd.DataFrame(items, columns=cols)
    df["age"] = age
    df["gender"] = gender
    df["jaundice"] = jaundice
    df["Class/ASD"] = ((items.sum(axis=1) >= 6).astype(int)).astype(str)
    df["Class/ASD"] = df["Class/ASD"].map({"1": "YES", "0": "NO"})
    source = "synthetic (mimics scoring rule)"

print(f"source: {source}  shape: {df.shape}")
df.head(3)

source: synthetic (mimics scoring rule)  shape: (800, 14)


,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,age,gender,jaundice,Class/ASD
0,1,0,0,0,0,1,0,0,0,1,18,m,no,NO
1,1,1,1,0,1,0,1,0,0,1,56,m,yes,YES
2,0,1,0,0,1,0,1,1,1,0,21,m,yes,NO


In [3]:
# normalise the label
label_col = "Class/ASD"
df[label_col] = df[label_col].astype(str).str.upper().map({"YES": 1, "NO": 0}).fillna(df[label_col])
y = df[label_col].astype(int).values

score_cols = [c for c in df.columns if c.startswith("A") and c.endswith("_Score")]
num_cols = score_cols + (["age"] if "age" in df.columns else [])
cat_cols = [c for c in ["gender", "jaundice", "ethnicity", "country_of_res"] if c in df.columns]

X_df = df[num_cols + cat_cols].copy()
for c in cat_cols:
    X_df[c] = X_df[c].fillna("unknown").astype(str)

X_df = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
print("final feature count:", X_df.shape[1])

final feature count: 13


In [4]:
X = X_df.values.astype(float)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

for name, clf in [
    ("LogReg", Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))])),
    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=2)),
]:
    auc = cross_val_score(clf, X, y, cv=skf, scoring="roc_auc").mean()
    print(f"{name:13s}  AUC: {auc:.3f}")

# train one model on a holdout for the report
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=2)
final = Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)
print(classification_report(y_te, final.predict(X_te), digits=3))

LogReg         AUC: 1.000


RandomForest   AUC: 0.976
              precision    recall  f1-score   support

           0      1.000     1.000     1.000       101
           1      1.000     1.000     1.000        59

    accuracy                          1.000       160
   macro avg      1.000     1.000     1.000       160
weighted avg      1.000     1.000     1.000       160



## Takeaways

AUC sits near 1.0 because the AQ-10 score is, by definition, a deterministic function of the 10 item responses (sum >= 6). The model is basically recovering that rule. The interesting question is what the model does with the *demographics* — they shouldn't add much, and if they do, it's worth checking whether there's bias in who got screened positive in the first place.

This is a useful exercise but I wouldn't oversell it. The dataset doesn't contain a real diagnostic ground truth, and the AQ-10 is a screening tool with known false-positive issues, particularly in adults with anxiety or ADHD. A model on top of a screener inherits all of the screener's limitations.